# Exploracao do PlantVillage

Auditoria inicial do dataset PlantVillage usando apenas as imagens em `raw/color/`, seguida da divisao train/validation/test nos metadados, sem extrair ou copiar imagens e sem treinamento.

In [1]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

In [2]:
if IN_COLAB:
    BASE_DIR = Path("/content/drive/MyDrive/TCC")
else:
    current_dir = Path.cwd().resolve()
    BASE_DIR = current_dir.parent if current_dir.name == "notebooks" else current_dir

DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
SRC_DIR = BASE_DIR / "src"

ZIP_PATH = DATA_DIR / "data.zip"
LEAF_MAP_PATH = DATA_DIR / "leaf_grouping" / "leaf-map.json"
OUTPUT_CSV = RESULTS_DIR / "plantvillage_metadata_raw_color.csv"
SPLIT_OUTPUT_CSV = RESULTS_DIR / "plantvillage_metadata_split.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Base:", BASE_DIR)
print("ZIP:", ZIP_PATH)
print("Leaf map:", LEAF_MAP_PATH)
print("Resultados:", RESULTS_DIR)

Base: C:\Users\cw_58\Documents\tcc-plant-disease-classification
ZIP: C:\Users\cw_58\Documents\tcc-plant-disease-classification\data\data.zip
Leaf map: C:\Users\cw_58\Documents\tcc-plant-disease-classification\data\leaf_grouping\leaf-map.json
Resultados: C:\Users\cw_58\Documents\tcc-plant-disease-classification\results


## Arquivos oficiais

In [3]:
try:
    from huggingface_hub import hf_hub_download
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import hf_hub_download

In [4]:
if not ZIP_PATH.exists():
    ZIP_PATH = Path(
        hf_hub_download(
            repo_id="mohanty/PlantVillage",
            filename="data.zip",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
        )
    )

if not LEAF_MAP_PATH.exists():
    LEAF_MAP_PATH = Path(
        hf_hub_download(
            repo_id="mohanty/PlantVillage",
            filename="leaf_grouping/leaf-map.json",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
        )
    )

print("data.zip existe:", ZIP_PATH.exists())
print("leaf-map.json existe:", LEAF_MAP_PATH.exists())

if ZIP_PATH.exists():
    print(f"Tamanho do data.zip: {ZIP_PATH.stat().st_size / (1024 ** 3):.2f} GB")

data.zip existe: True
leaf-map.json existe: True
Tamanho do data.zip: 2.03 GB


## Metadados

In [5]:
import importlib.util
import sys
import urllib.request

module_file = SRC_DIR / "plantvillage_audit.py"
required_markers = (
    "FALLBACK_PREFIX",
    "leaf_id_source",
    "origem_leaf_id",
    "numero_identificadores_agrupamento_unicos",
    "maiores_grupos_leaf_id",
    "maiores_grupos_fallback",
    "fallbacks_multiclasse",
    ".replace(\".JPG\", \"\")",
)

module_text = module_file.read_text(encoding="utf-8") if module_file.exists() else ""
if not all(marker in module_text for marker in required_markers):
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    module_url = "https://raw.githubusercontent.com/murilodc/plant-disease-classification/main/src/plantvillage_audit.py"
    module_text = urllib.request.urlopen(module_url).read().decode("utf-8")
    module_file.write_text(module_text, encoding="utf-8")
    print("Modulo atualizado em:", module_file)

module_text = module_file.read_text(encoding="utf-8")
missing_markers = [marker for marker in required_markers if marker not in module_text]
if missing_markers:
    raise RuntimeError(f"Modulo PlantVillage ainda esta desatualizado: {module_file}")

spec = importlib.util.spec_from_file_location("plantvillage_audit_runtime", module_file)
if spec is None or spec.loader is None:
    raise ImportError(f"Nao foi possivel carregar o modulo: {module_file}")

pv_audit = importlib.util.module_from_spec(spec)
sys.modules["plantvillage_audit_runtime"] = pv_audit
spec.loader.exec_module(pv_audit)

audit_metadata = pv_audit.audit_metadata
build_metadata_dataframe = pv_audit.build_metadata_dataframe
save_metadata_csv = pv_audit.save_metadata_csv

print("Modulo carregado de arquivo:", pv_audit.__file__)

fallback_test = pv_audit.resolve_leaf_id("x.JPG", "Classe___Teste", {})
if fallback_test.get("leaf_id") != "fallback_x" or fallback_test.get("leaf_id_source") != "fallback":
    raise RuntimeError(f"Modulo PlantVillage desatualizado: {pv_audit.__file__}")

metadata = build_metadata_dataframe(
    zip_path=ZIP_PATH,
    leaf_map_path=LEAF_MAP_PATH,
)

print("Formato do DataFrame:", metadata.shape)
metadata.head()

Modulo carregado de arquivo: C:\Users\cw_58\Documents\tcc-plant-disease-classification\src\plantvillage_audit.py


Formato do DataFrame: (54305, 11)


,zip_path,filename,classe,cultura,doenca,leaf_id,leaf_id_found,leaf_id_source,leaf_match_status,leaf_lookup_key,leaf_suggestions_count
0,raw/color/Raspberry___healthy/6c2e049f-38c9-42...,6c2e049f-38c9-42c5-864f-613e6e12d59e___Mary_HL...,Raspberry___healthy,Raspberry,healthy,Raspberry___healthy:::11.0,True,leaf-map,matched_unique,mary_hl 6318,1
1,raw/color/Raspberry___healthy/d1387960-14d6-46...,d1387960-14d6-4654-ad5c-b92afa86ff6a___Mary_HL...,Raspberry___healthy,Raspberry,healthy,Raspberry___healthy:::32.0,True,leaf-map,matched_unique,mary_hl 9262,1
2,raw/color/Raspberry___healthy/fafac0a9-e5ba-42...,fafac0a9-e5ba-420d-9dbe-034942f845b9___Mary_HL...,Raspberry___healthy,Raspberry,healthy,Raspberry___healthy:::28.0,True,leaf-map,matched_unique,mary_hl 9201,1
3,raw/color/Raspberry___healthy/a37f34f7-022d-46...,a37f34f7-022d-461a-8a3d-95f5cd774e35___Mary_HL...,Raspberry___healthy,Raspberry,healthy,Raspberry___healthy:::25.0,True,leaf-map,matched_unique,mary_hl 9155,1
4,raw/color/Raspberry___healthy/fe1c1683-06cc-49...,fe1c1683-06cc-49c5-b86a-a41fb36f058b___Mary_HL...,Raspberry___healthy,Raspberry,healthy,Raspberry___healthy:::5.0,True,leaf-map,matched_unique,mary_hl 6249,1


## Resumo da auditoria

In [6]:
auditoria = audit_metadata(metadata)

print("Chaves da auditoria:", list(auditoria))

auditoria["resumo"]

Chaves da auditoria: ['resumo', 'imagens_por_classe', 'origem_leaf_id', 'status_leaf_id', 'maiores_grupos_leaf_id', 'maiores_grupos_fallback', 'fallback_r_imagens', 'fallback_r_imagens_por_classe', 'fallbacks_multiclasse']


,total_imagens,numero_classes,numero_culturas,numero_doencas,imagens_associadas_leaf_map,imagens_usando_fallback,percentual_usando_fallback,numero_identificadores_agrupamento_unicos,imagens_em_grupos_com_multiplas_imagens,grupos_com_multiplas_imagens,fallbacks_unicos,fallbacks_com_mais_de_uma_imagem,fallbacks_em_mais_de_uma_classe
0,54305,38,14,26,41111,13194,24.2961,20015,42471,8181,12422,600,0


## Origem dos identificadores

In [7]:
auditoria["origem_leaf_id"]

,leaf_id_source,quantidade
0,fallback,13194
1,leaf-map,41111


In [8]:
associadas_leaf_map = metadata.loc[metadata["leaf_id_source"].eq("leaf-map")]
fallbacks = metadata.loc[metadata["leaf_id_source"].eq("fallback")]

print("Imagens associadas pelo leaf-map:", len(associadas_leaf_map))
print("Imagens usando fallback:", len(fallbacks))

Imagens associadas pelo leaf-map: 41111
Imagens usando fallback: 13194


## Status da associacao

In [9]:
auditoria["status_leaf_id"]

,leaf_match_status,quantidade
0,matched_by_class,4370
1,matched_unique,36741
2,not_found,13194


## Maiores grupos leaf_id

In [10]:
auditoria["maiores_grupos_leaf_id"]

,leaf_id,quantidade_imagens,quantidade_classes,leaf_id_source
0,Soybean___healthy:::367.0,33,1,leaf-map
1,Blueberry___healthy:::144.0,29,1,leaf-map
2,Soybean___healthy:::368.0,28,1,leaf-map
3,Orange___Haunglongbing_(Citrus_greening):::47.0,25,1,leaf-map
4,Blueberry___healthy:::152.0,24,1,leaf-map
5,Cherry_(including_sour)___Powdery_mildew:::11.0,24,1,leaf-map
6,Cherry_(including_sour)___Powdery_mildew:::35.0,24,1,leaf-map
7,Cherry_(including_sour)___Powdery_mildew:::43.0,24,1,leaf-map
8,Orange___Haunglongbing_(Citrus_greening):::36.0,24,1,leaf-map
9,Cherry_(including_sour)___Powdery_mildew:::23.0,23,1,leaf-map


## Maiores grupos fallback

In [11]:
auditoria["maiores_grupos_fallback"]

,leaf_id,quantidade_imagens,quantidade_classes,leaf_id_source
0,fallback_r.s_hl 0632,5,1,fallback
1,fallback_r.s_hl 0646,5,1,fallback
2,fallback_r.s_hl 0596,4,1,fallback
3,fallback_r.s_hl 0598,4,1,fallback
4,fallback_r.s_hl 0599,4,1,fallback
5,fallback_r.s_hl 0600,4,1,fallback
6,fallback_r.s_hl 0601,4,1,fallback
7,fallback_r.s_hl 0609,4,1,fallback
8,fallback_r.s_hl 0623,4,1,fallback
9,fallback_r.s_hl 0626,4,1,fallback


## Colisoes de fallback

In [12]:
resumo = auditoria["resumo"].iloc[0]

print("Fallbacks com mais de uma imagem:", resumo["fallbacks_com_mais_de_uma_imagem"])
print("Fallbacks em mais de uma classe:", resumo["fallbacks_em_mais_de_uma_classe"])

Fallbacks com mais de uma imagem: 600.0
Fallbacks em mais de uma classe: 0.0


## Grupo fallback_r

In [13]:
fallback_r = auditoria["fallback_r_imagens"]

print("Imagens no grupo fallback_r:", len(fallback_r))
if len(fallback_r) > 0:
    print("Classes no grupo fallback_r:", fallback_r["quantidade_classes_no_grupo"].iloc[0])
else:
    print("Classes no grupo fallback_r: 0")

fallback_r.head(20)

Imagens no grupo fallback_r: 0
Classes no grupo fallback_r: 0


,filename,classe,leaf_lookup_key,leaf_id,quantidade_classes_no_grupo,quantidade_imagens_na_classe


In [14]:
auditoria["fallback_r_imagens_por_classe"]

,classe,quantidade


## Fallbacks em mais de uma classe

In [15]:
auditoria["fallbacks_multiclasse"]

,leaf_id,quantidade_imagens,quantidade_classes,classes


## Imagens por classe

In [16]:
auditoria["imagens_por_classe"]

,classe,quantidade
0,Apple___Apple_scab,630
1,Apple___Black_rot,621
2,Apple___Cedar_apple_rust,275
3,Apple___healthy,1645
4,Blueberry___healthy,1502
5,Cherry_(including_sour)___Powdery_mildew,1052
6,Cherry_(including_sour)___healthy,854
7,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,513
8,Corn_(maize)___Common_rust_,1192
9,Corn_(maize)___Northern_Leaf_Blight,985


## Salvar metadados brutos

In [17]:
csv_path = save_metadata_csv(metadata, OUTPUT_CSV)

print("CSV salvo em:", csv_path)
print("Arquivo existe:", csv_path.exists())

CSV salvo em: C:\Users\cw_58\Documents\tcc-plant-disease-classification\results\plantvillage_metadata_raw_color.csv
Arquivo existe: True


## Divisao train/validation/test

In [18]:
import importlib.util
import sys
from pathlib import Path

if "SRC_DIR" not in globals():
    current_dir = Path.cwd().resolve()
    BASE_DIR = current_dir.parent if current_dir.name == "notebooks" else current_dir
    SRC_DIR = BASE_DIR / "src"
    RESULTS_DIR = BASE_DIR / "results"
    SPLIT_OUTPUT_CSV = RESULTS_DIR / "plantvillage_metadata_split.csv"

split_module_file = SRC_DIR / "plantvillage_split.py"
if not split_module_file.exists():
    raise FileNotFoundError(
        f"Nao encontrei o modulo de split em {split_module_file}. "
        "Atualize a pasta src/ do projeto antes de executar esta celula."
    )

spec = importlib.util.spec_from_file_location("plantvillage_split_runtime", split_module_file)
if spec is None or spec.loader is None:
    raise ImportError(f"Nao foi possivel carregar o modulo: {split_module_file}")

pv_split = importlib.util.module_from_spec(spec)
sys.modules["plantvillage_split_runtime"] = pv_split
spec.loader.exec_module(pv_split)

SPLIT_SEED = pv_split.SPLIT_SEED
save_split_metadata_csv = pv_split.save_split_metadata_csv
split_diagnostics = pv_split.split_diagnostics
split_metadata_by_leaf_id = pv_split.split_metadata_by_leaf_id
validate_leaf_id_single_class = pv_split.validate_leaf_id_single_class

print("Modulo de split carregado de arquivo:", pv_split.__file__)

validate_leaf_id_single_class(metadata)
metadata_split = split_metadata_by_leaf_id(metadata, seed=SPLIT_SEED)
validacoes_split = split_diagnostics(metadata_split)

if not validacoes_split["validacoes"]["ok"].all():
    raise RuntimeError("A divisao gerada nao passou em todas as validacoes.")

split_csv_path = save_split_metadata_csv(metadata_split, SPLIT_OUTPUT_CSV)

print("Seed:", SPLIT_SEED)
print("CSV com split salvo em:", split_csv_path)
print("Arquivo existe:", split_csv_path.exists())

metadata_split.head()

Modulo de split carregado de arquivo: C:\Users\cw_58\Documents\tcc-plant-disease-classification\src\plantvillage_split.py


Seed: 42
CSV com split salvo em: C:\Users\cw_58\Documents\tcc-plant-disease-classification\results\plantvillage_metadata_split.csv
Arquivo existe: True


,zip_path,filename,classe,cultura,doenca,leaf_id,leaf_id_found,leaf_id_source,leaf_match_status,leaf_lookup_key,leaf_suggestions_count,split
0,raw/color/Raspberry___healthy/6c2e049f-38c9-42...,6c2e049f-38c9-42c5-864f-613e6e12d59e___Mary_HL...,Raspberry___healthy,Raspberry,healthy,Raspberry___healthy:::11.0,True,leaf-map,matched_unique,mary_hl 6318,1,test
1,raw/color/Raspberry___healthy/d1387960-14d6-46...,d1387960-14d6-4654-ad5c-b92afa86ff6a___Mary_HL...,Raspberry___healthy,Raspberry,healthy,Raspberry___healthy:::32.0,True,leaf-map,matched_unique,mary_hl 9262,1,train
2,raw/color/Raspberry___healthy/fafac0a9-e5ba-42...,fafac0a9-e5ba-420d-9dbe-034942f845b9___Mary_HL...,Raspberry___healthy,Raspberry,healthy,Raspberry___healthy:::28.0,True,leaf-map,matched_unique,mary_hl 9201,1,train
3,raw/color/Raspberry___healthy/a37f34f7-022d-46...,a37f34f7-022d-461a-8a3d-95f5cd774e35___Mary_HL...,Raspberry___healthy,Raspberry,healthy,Raspberry___healthy:::25.0,True,leaf-map,matched_unique,mary_hl 9155,1,train
4,raw/color/Raspberry___healthy/fe1c1683-06cc-49...,fe1c1683-06cc-49c5-b86a-a41fb36f058b___Mary_HL...,Raspberry___healthy,Raspberry,healthy,Raspberry___healthy:::5.0,True,leaf-map,matched_unique,mary_hl 6249,1,validation


## Quantidade e percentual de imagens por split

In [19]:
validacoes_split["imagens_por_split"]

,split,quantidade_imagens,percentual_imagens
0,train,38008,69.9899
1,validation,8172,15.0483
2,test,8125,14.9618


## Quantidade de leaf_id por split

In [20]:
validacoes_split["leaf_ids_por_split"]

,split,quantidade_leaf_id
0,train,12339
1,validation,3842
2,test,3834


## Quantidade de imagens de cada classe em cada split

In [21]:
validacoes_split["imagens_por_classe_split"]

split,train,validation,test
classe,,,
Apple___Apple_scab,440,95,95
Apple___Black_rot,436,95,90
Apple___Cedar_apple_rust,192,43,40
Apple___healthy,1151,247,247
Blueberry___healthy,1051,227,224
Cherry_(including_sour)___Powdery_mildew,736,158,158
Cherry_(including_sour)___healthy,598,130,126
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot,359,77,77
Corn_(maize)___Common_rust_,834,179,179


## Percentual de cada classe destinado a cada split

In [22]:
validacoes_split["percentual_classe_por_split"]

split,train,validation,test
classe,,,
Apple___Apple_scab,69.8413,15.0794,15.0794
Apple___Black_rot,70.2093,15.2979,14.4928
Apple___Cedar_apple_rust,69.8182,15.6364,14.5455
Apple___healthy,69.9696,15.0152,15.0152
Blueberry___healthy,69.9734,15.1132,14.9134
Cherry_(including_sour)___Powdery_mildew,69.9620,15.0190,15.0190
Cherry_(including_sour)___healthy,70.0234,15.2225,14.7541
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot,69.9805,15.0097,15.0097
Corn_(maize)___Common_rust_,69.9664,15.0168,15.0168


## Classes por split

In [23]:
validacoes_split["classes_por_split"]

,split,quantidade_classes
0,train,38
1,validation,38
2,test,38


## Confirmacoes da divisao

In [24]:
validacoes_split["validacoes"]

,validacao,ok,detalhe
0,nenhum_leaf_id_em_mais_de_um_split,True,0 leaf_id repetidos
1,todas_as_54305_imagens_atribuidas,True,54305 imagens; 0 sem split; 0 splits invalidos
2,as_38_classes_aparecem_nos_tres_splits,True,38 classes no total; minimo por split: 38
